In [2]:

import pandas as pd
import os, glob

DATA = "/workspace/value_up_ai/data"

files = {
    "buyer_db": "buyer_db.csv",
    "kotra_sns": "kotra_sns_buyers.csv",
    "kotra_inquiry": "kotra_inquiry.csv",
    "kotra_recommend": "kotra_hs_country_recommend.csv",
    "kotra_buyer_stats": "kotra_buyer_stats.csv",
    "smba_inquiry": "smba_inquiry.csv",
    "smba_offer": "smba_purchase_offer.csv",
    "nipa_ict": "nipa_ict_buyers.csv",
    "ksure_email": "ksure_cosmetic_email_verified.csv",
    "ksure_full": "ksure_cosmetic_buyers_full.csv",
    "aT_bms": "aT_bms_buyers.csv",
    "trade_regulation": "trade_regulation_db.csv",
    "country_credit": "country_credit_db.csv",
    "email_pattern": "email_pattern_db.csv",
}

results = {}
for key, fname in files.items():
    path = os.path.join(DATA, fname)
    if not os.path.exists(path):
        continue
    for enc in ["utf-8-sig", "utf-8", "euc-kr"]:
        try:
            df = pd.read_csv(path, encoding=enc, dtype=str)
            sample = df.head(2).to_dict("records")
            results[key] = {
                "rows": len(df),
                "columns": list(df.columns),
                "sample": sample,
                "null_pct": {c: round(df[c].isna().mean()*100,1) for c in df.columns},
            }
            break
        except:
            continue

import json
print(json.dumps(results, ensure_ascii=False, indent=2)[:8000])


{
  "buyer_db": {
    "rows": 46089,
    "columns": [
      "hs_code",
      "country",
      "buyer_name",
      "annual_usd",
      "shipments",
      "last_date",
      "buyer_type",
      "city",
      "source"
    ],
    "sample": [
      {
        "hs_code": "330499",
        "country": "VN",
        "buyer_name": "Công ty TNHH Mỹ Phẩm Sài Gòn",
        "annual_usd": "1680000",
        "shipments": "18",
        "last_date": "2026-02-20",
        "buyer_type": "Distributor",
        "city": "Ho Chi Minh",
        "source": "Customs_VN"
      },
      {
        "hs_code": "330499",
        "country": "VN",
        "buyer_name": "Hanoi Beauty & Wellness Corp",
        "annual_usd": "1140000",
        "shipments": "14",
        "last_date": "2026-03-01",
        "buyer_type": "Wholesaler",
        "city": "Hanoi",
        "source": "Customs_VN"
      }
    ],
    "null_pct": {
      "hs_code": 0.0,
      "country": 0.0,
      "buyer_name": 0.5,
      "annual_usd": 99.9,
      "shipm

In [5]:

import pandas as pd
import os

DATA = "/workspace/value_up_ai/data"

print("=" * 60)
print("📊 현재 데이터 한계점 정량 분석")
print("=" * 60)

# 1. buyer_db: annual_usd 결측률
df_buyer = pd.read_csv(f"{DATA}/buyer_db.csv", dtype=str)
print(f"\n[buyer_db.csv] {len(df_buyer):,}행")
null_annual = df_buyer['annual_usd'].isna().mean() * 100
null_email = 100.0  # 이메일 컬럼 없음
print(f"  ❌ annual_usd 결측: {null_annual:.1f}%")
print(f"  ❌ 이메일 컬럼: 없음")
print(f"  ❌ 회사 연락처: 없음")
print(f"  ❌ 최근거래일(last_date) 결측: {df_buyer['last_date'].isna().mean()*100:.1f}%")
print(f"  국가 분포 Top5: {df_buyer['country'].value_counts().head(5).to_dict()}")

# 2. KOTRA 추천: ISO2 결측률
df_kotra = pd.read_csv(f"{DATA}/kotra_hs_country_recommend.csv", dtype=str)
print(f"\n[kotra_hs_country_recommend.csv] {len(df_kotra):,}행")
print(f"  ❌ country_iso 결측: {df_kotra['country_iso'].isna().mean()*100:.1f}%")
hs_set = df_kotra['hs_code'].unique()
print(f"  ⚠️ HS코드 종류: {len(hs_set)}개 (330499 하나만?)")
print(f"  HS코드 목록: {list(hs_set)[:10]}")

# 3. KOTRA 인콰이어리: 바이어명 없음
df_inq = pd.read_csv(f"{DATA}/kotra_inquiry.csv", dtype=str)
print(f"\n[kotra_inquiry.csv] {len(df_inq):,}행")
print(f"  ❌ 바이어명(company): 없음")
print(f"  ❌ 이메일: 없음")
print(f"  ❌ country ISO 결측: {df_inq['country'].isna().mean()*100:.1f}%")
print(f"  ❌ HS코드: 없음")
top_product = df_inq['product_en'].value_counts().head(5)
print(f"  상위 제품요청: {top_product.to_dict()}")

# 4. smba_inquiry: HS코드 없음
df_smba = pd.read_csv(f"{DATA}/smba_inquiry.csv", dtype=str)
print(f"\n[smba_inquiry.csv] {len(df_smba):,}행")
print(f"  ❌ HS코드: 없음")
print(f"  ❌ 바이어명: 없음")
print(f"  ❌ 이메일: 없음")
print(f"  ❌ country 결측: {df_smba['country'].isna().mean()*100:.1f}%")

# 5. nipa_ict: 국가명만 있고 ISO코드 없음, 업종 없음
df_nipa = pd.read_csv(f"{DATA}/nipa_ict_buyers.csv", dtype=str)
print(f"\n[nipa_ict_buyers.csv] {len(df_nipa):,}행")
print(f"  ❌ 국가 ISO코드: 없음 (국가명만)")
print(f"  ❌ 이메일: 없음 (전화번호만)")
print(f"  ❌ HS코드/업종: 없음")
print(f"  국가분포 Top5: {df_nipa['nationName'].value_counts().head(5).to_dict()}")

# 6. ksure_email: 국가 컬럼 없음
df_ksure = pd.read_csv(f"{DATA}/ksure_cosmetic_email_verified.csv", dtype=str)
print(f"\n[ksure_cosmetic_email_verified.csv] {len(df_ksure):,}행")
print(f"  ❌ 국가 ISO코드: 없음 (주소에서 추정 필요)")
print(f"  ❌ HS코드: 없음 (업종코드만)")
print(f"  업종 분포: {df_ksure['업종한글명'].value_counts().to_dict()}")

# 7. trade_regulation: HS코드 컬럼 파악
df_reg = pd.read_csv(f"{DATA}/trade_regulation_db.csv", dtype=str)
print(f"\n[trade_regulation_db.csv] {len(df_reg):,}행")
print(f"  컬럼: {list(df_reg.columns)}")
print(f"  샘플: {df_reg.iloc[0].to_dict()}")


📊 현재 데이터 한계점 정량 분석

[buyer_db.csv] 46,089행
  ❌ annual_usd 결측: 99.9%
  ❌ 이메일 컬럼: 없음
  ❌ 회사 연락처: 없음
  ❌ 최근거래일(last_date) 결측: 99.9%
  국가 분포 Top5: {'IN': 8277, 'US': 4571, 'PK': 2687, 'PH': 2683, 'AR': 2564}

[kotra_hs_country_recommend.csv] 2,100행
  ❌ country_iso 결측: 71.3%
  ⚠️ HS코드 종류: 7개 (330499 하나만?)
  HS코드 목록: ['330499', '870830', '210690', '330410', '330510', '330590', '300490']

[kotra_inquiry.csv] 40,305행
  ❌ 바이어명(company): 없음
  ❌ 이메일: 없음
  ❌ country ISO 결측: 3.6%
  ❌ HS코드: 없음
  상위 제품요청: {'Tomato Orthopedic Casting Tape': 76, 'MAUVE COVID19 Ag Test': 27, 'power_item01': 18, 'Good Pencil': 17, 'Face Cream': 15}



[smba_inquiry.csv] 21,302행
  ❌ HS코드: 없음
  ❌ 바이어명: 없음
  ❌ 이메일: 없음
  ❌ country 결측: 15.6%

[nipa_ict_buyers.csv] 1,853행
  ❌ 국가 ISO코드: 없음 (국가명만)
  ❌ 이메일: 없음 (전화번호만)
  ❌ HS코드/업종: 없음
  국가분포 Top5: {'아랍에미리트/두바이': 68, 'UAE / Dubai': 55, '미국': 40, '싱가포르': 34, '인도': 34}

[ksure_cosmetic_email_verified.csv] 214행
  ❌ 국가 ISO코드: 없음 (주소에서 추정 필요)
  ❌ HS코드: 없음 (업종코드만)
  업종 분포: {'화장품및화장용품도매업': 121, '화장품제조업': 69, '화장품,비누및방향제소매업': 24}

[trade_regulation_db.csv] 27,959행
  컬럼: ['regulation_country', 'product_name', 'regulation_type', 'target_country', 'tariff_rate', 'hs_code_6']
  샘플: {'regulation_country': 'AE', 'product_name': '납축전지(자동차배터리)(Electric lead-acid accumulators)(Automotive batteries of capacity from 35 to 115 ambers)', 'regulation_type': '반덤핑(규제중)', 'target_country': '한국', 'tariff_rate': 'ㅇ 판정결과 : 12 ~ 25%* 2023.4.11일 일몰재심 최종판정: 규제 5년 연장 결정', 'hs_code_6': '850710'}


In [8]:

import pandas as pd
import os

DATA = "/workspace/value_up_ai/data"

# ─────────────────────────────────────────────────────────
# 보완 1: kotra_recommend country_iso 71.3% 결측 → 매핑 보완
# ─────────────────────────────────────────────────────────
KO_ISO2 = {
    '가나':'GH','나이지리아':'NG','남아프리카공화국':'ZA','에티오피아':'ET','케냐':'KE',
    '탄자니아':'TZ','모로코':'MA','이집트':'EG','세네갈':'SN','앙골라':'AO',
    '코트디부아르':'CI','카메룬':'CM','가봉':'GA','짐바브웨':'ZW','모잠비크':'MZ',
    '르완다':'RW','우간다':'UG','가나':'GH','에콰도르':'EC','과테말라':'GT',
    '도미니카공화국':'DO','온두라스':'HN','파나마':'PA','볼리비아':'BO','파라과이':'PY',
    '우루과이':'UY','코스타리카':'CR','엘살바도르':'SV','니카라과':'NI',
    '트리니다드토바고':'TT','자메이카':'JM','쿠바':'CU','아이티':'HT',
    '이란':'IR','이라크':'IQ','이스라엘':'IL','요르단':'JO','레바논':'LB',
    '쿠웨이트':'KW','카타르':'QA','바레인':'BH','오만':'OM','예멘':'YE',
    '시리아':'SY','리비아':'LY','튀니지':'TN','알제리':'DZ','모리타니':'MR',
    '파키스탄':'PK','방글라데시':'BD','스리랑카':'LK','네팔':'NP','미얀마':'MM',
    '캄보디아':'KH','라오스':'LA','몽골':'MN','카자흐스탄':'KZ','우즈베키스탄':'UZ',
    '투르크메니스탄':'TM','타지키스탄':'TJ','키르기스스탄':'KG','아제르바이잔':'AZ',
    '조지아':'GE','아르메니아':'AM','우크라이나':'UA','폴란드':'PL','체코':'CZ',
    '슬로바키아':'SK','헝가리':'HU','루마니아':'RO','불가리아':'BG','크로아티아':'HR',
    '세르비아':'RS','슬로베니아':'SI','에스토니아':'EE','라트비아':'LV','리투아니아':'LT',
    '핀란드':'FI','노르웨이':'NO','덴마크':'DK','스웨덴':'SE','스위스':'CH',
    '오스트리아':'AT','벨기에':'BE','네덜란드':'NL','포르투갈':'PT','그리스':'GR',
    '아일랜드':'IE','뉴질랜드':'NZ','파푸아뉴기니':'PG','피지':'FJ',
    '미국':'US','영국':'GB','중국':'CN','일본':'JP','독일':'DE',
    '프랑스':'FR','베트남':'VN','태국':'TH','싱가포르':'SG','말레이시아':'MY',
    '인도네시아':'ID','인도':'IN','호주':'AU','캐나다':'CA','이탈리아':'IT',
    '스페인':'ES','홍콩':'HK','대만':'TW','아랍에미리트':'AE','사우디아라비아':'SA',
    '브라질':'BR','멕시코':'MX','아르헨티나':'AR','콜롬비아':'CO','칠레':'CL',
    '페루':'PE','터키':'TR','러시아':'RU','미국령사모아':'AS',
}

df_kotra = pd.read_csv(f"{DATA}/kotra_hs_country_recommend.csv", dtype=str)
before_null = df_kotra['country_iso'].isna().sum()
df_kotra['country_iso'] = df_kotra.apply(
    lambda r: r['country_iso'] if pd.notna(r['country_iso']) and r['country_iso']
    else KO_ISO2.get(r['country_name'], ''),
    axis=1
)
df_kotra['country_iso'] = df_kotra['country_iso'].replace('', pd.NA)
after_null = df_kotra['country_iso'].isna().sum()
df_kotra.to_csv(f"{DATA}/kotra_hs_country_recommend.csv", index=False, encoding='utf-8-sig')
print(f"✅ kotra_recommend country_iso 결측: {before_null}→{after_null}건")
print(f"   커버리지: {100 - after_null/len(df_kotra)*100:.1f}%")

# ─────────────────────────────────────────────────────────
# 보완 2: nipa_ict 국가명→ISO2 추가
# ─────────────────────────────────────────────────────────
NIPA_NAME_MAP = {
    '아랍에미리트/두바이':'AE','UAE / Dubai':'AE','UAE':'AE',
    '미국':'US','싱가포르':'SG','인도':'IN','태국':'TH','베트남':'VN',
    '필리핀':'PH','말레이시아':'MY','인도네시아':'ID','중국':'CN','일본':'JP',
    '카자흐스탄':'KZ','러시아':'RU','호주':'AU','캐나다':'CA','영국':'GB',
    '독일':'DE','프랑스':'FR','브라질':'BR','남아프리카공화국':'ZA','나이지리아':'NG',
    '이집트':'EG','케냐':'KE','가나':'GH','사우디아라비아':'SA','이란':'IR',
    '파키스탄':'PK','방글라데시':'BD','스리랑카':'LK','홍콩':'HK','대만':'TW',
    '멕시코':'MX','아르헨티나':'AR','콜롬비아':'CO','페루':'PE','칠레':'CL',
    '터키':'TR','이스라엘':'IL','카타르':'QA','쿠웨이트':'KW','오만':'OM',
    '우즈베키스탄':'UZ','몽골':'MN','캄보디아':'KH','미얀마':'MM','라오스':'LA',
    '스웨덴':'SE','노르웨이':'NO','핀란드':'FI','덴마크':'DK','네덜란드':'NL',
    '폴란드':'PL','루마니아':'RO','체코':'CZ','헝가리':'HU','우크라이나':'UA',
}

df_nipa = pd.read_csv(f"{DATA}/nipa_ict_buyers.csv", dtype=str)
df_nipa['country_iso'] = df_nipa['nationName'].map(NIPA_NAME_MAP).fillna('')
mapped = (df_nipa['country_iso'] != '').sum()
df_nipa.to_csv(f"{DATA}/nipa_ict_buyers.csv", index=False, encoding='utf-8-sig')
print(f"✅ nipa_ict country_iso 추가: {mapped}/{len(df_nipa)}건 매핑 ({mapped/len(df_nipa)*100:.1f}%)")

# ─────────────────────────────────────────────────────────
# 보완 3: ksure_email 주소→국가 추정
# ─────────────────────────────────────────────────────────
COUNTRY_HINT = {
    'NIGERIA':'NG','SOUTHAFRICA':'ZA','SINGAPORE':'SG','MALAYSIA':'MY',
    'INDONESIA':'ID','VIETNAM':'VN','THAILAND':'TH','PHILIPPINES':'PH',
    'INDIA':'IN','CHINA':'CN','JAPAN':'JP','USA':'US','GERMANY':'DE',
    'FRANCE':'FR','BRAZIL':'BR','MEXICO':'MX','EGYPT':'EG','KENYA':'KE',
    'GHANA':'GH','TANZANIA':'TZ','AUSTRALIA':'AU','CANADA':'CA',
    'UNITEDKINGDOM':'GB','UK':'GB','UAE':'AE','DUBAI':'AE',
    'KOREA':'KR','SAUDIARABIA':'SA','TURKEY':'TR','RUSSIA':'RU',
}

df_ksure = pd.read_csv(f"{DATA}/ksure_cosmetic_email_verified.csv", dtype=str)
def guess_country(row):
    addr = str(row.get('주소','')).upper().replace(' ','').replace(',','')
    for hint, iso in COUNTRY_HINT.items():
        if hint in addr:
            return iso
    return ''

df_ksure['country_iso'] = df_ksure.apply(guess_country, axis=1)
mapped_k = (df_ksure['country_iso'] != '').sum()
df_ksure.to_csv(f"{DATA}/ksure_cosmetic_email_verified.csv", index=False, encoding='utf-8-sig')
print(f"✅ ksure_email country_iso 추가: {mapped_k}/{len(df_ksure)}건 ({mapped_k/len(df_ksure)*100:.1f}%)")
print(f"   국가분포: {df_ksure['country_iso'].value_counts().head(8).to_dict()}")

# ─────────────────────────────────────────────────────────
# 보완 4: KOTRA 인콰이어리 제품명→HS코드 추정 (4자리 prefix)
# ─────────────────────────────────────────────────────────
HS_KEYWORD_MAP = {
    'cosmetic':  '3304', 'beauty':    '3304', 'skincare':  '3304', 'cream':     '3304',
    'lipstick':  '3304', 'makeup':    '3304', 'perfume':   '3303', 'shampoo':   '3305',
    'soap':      '3401', 'medicine':  '3004', 'drug':      '3004', 'supplement':'2106',
    'food':      '2106', 'snack':     '1905', 'beverage':  '2202', 'coffee':    '0901',
    'tea':       '0902', 'fruit':     '0804', 'vegetable': '0709', 'meat':      '0201',
    'fish':      '0302', 'textile':   '5407', 'fabric':    '5208', 'clothing':  '6109',
    'shoe':      '6403', 'bag':       '4202', 'electronic':'8517', 'phone':     '8517',
    'computer':  '8471', 'battery':   '8507', 'led':       '9405', 'solar':     '8541',
    'car':       '8703', 'auto':      '8703', 'tire':      '4011', 'machine':   '8479',
    'medical':   '9018', 'tape':      '3005', 'plastic':   '3926', 'steel':     '7208',
    'aluminum':  '7606', 'chemical':  '2915', 'fiber':     '5503',
}

df_inq = pd.read_csv(f"{DATA}/kotra_inquiry.csv", dtype=str)
def guess_hs(product_en):
    if pd.isna(product_en): return ''
    p = str(product_en).lower()
    for kw, hs in HS_KEYWORD_MAP.items():
        if kw in p:
            return hs
    return ''

df_inq['hs_prefix'] = df_inq['product_en'].apply(guess_hs)
matched_hs = (df_inq['hs_prefix'] != '').sum()
df_inq.to_csv(f"{DATA}/kotra_inquiry.csv", index=False, encoding='utf-8-sig')
print(f"✅ kotra_inquiry HS코드 추정: {matched_hs:,}/{len(df_inq):,}건 ({matched_hs/len(df_inq)*100:.1f}%)")

# ─────────────────────────────────────────────────────────
# 보완 5: smba_inquiry 제품명→HS코드 추정
# ─────────────────────────────────────────────────────────
KO_HS_MAP = {
    '화장품':'3304','뷰티':'3304','스킨케어':'3304','크림':'3304','마스크팩':'3304',
    '샴푸':'3305','향수':'3303','비누':'3401','의약품':'3004','건강기능식품':'2106',
    '식품':'2106','음료':'2202','농산물':'0709','수산물':'0302','의류':'6109',
    '섬유':'5407','신발':'6403','가방':'4202','전자':'8517','휴대폰':'8517',
    '컴퓨터':'8471','배터리':'8507','자동차':'8703','기계':'8479','의료기기':'9018',
    '스포츠':'9506','테이프':'3005','플라스틱':'3926','철강':'7208','화학':'2915',
    'LED':'9405','태양광':'8541','반도체':'8542','디스플레이':'8528',
}

df_smba = pd.read_csv(f"{DATA}/smba_inquiry.csv", dtype=str)
def guess_hs_ko(product_ko):
    if pd.isna(product_ko): return ''
    for kw, hs in KO_HS_MAP.items():
        if kw in str(product_ko):
            return hs
    return ''

df_smba['hs_prefix'] = df_smba['product_ko'].apply(guess_hs_ko)
matched_smba = (df_smba['hs_prefix'] != '').sum()
df_smba.to_csv(f"{DATA}/smba_inquiry.csv", index=False, encoding='utf-8-sig')
print(f"✅ smba_inquiry HS코드 추정: {matched_smba:,}/{len(df_smba):,}건 ({matched_smba/len(df_smba)*100:.1f}%)")
print("\n✅ 보완 5가지 완료!")


✅ kotra_recommend country_iso 결측: 1498→497건
   커버리지: 76.3%
✅ nipa_ict country_iso 추가: 770/1853건 매핑 (41.6%)
✅ ksure_email country_iso 추가: 112/214건 (52.3%)
   국가분포: {'': 102, 'RU': 18, 'TH': 17, 'VN': 11, 'IN': 11, 'SG': 9, 'ID': 8, 'GB': 6}


✅ kotra_inquiry HS코드 추정: 10,007/40,305건 (24.8%)
✅ smba_inquiry HS코드 추정: 4,944/21,302건 (23.2%)

✅ 보완 5가지 완료!


In [11]:

import requests

# 관세청 수출입 무역통계 API 테스트
API_KEY = "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23"

# 1. 관세청 HS코드 품목 분류 조회 API
print("=== 관세청 수출입 통계 API 테스트 ===")
try:
    url = "https://apis.data.go.kr/1220000/tradeStats/getTradeStatsList"
    params = {
        "serviceKey": API_KEY,
        "type": "json",
        "numOfRows": 5,
        "pageNo": 1,
        "yyyyMM": "202501",
        "hsCd": "3304",
    }
    r = requests.get(url, params=params, timeout=8)
    print(f"관세청 무역통계 → {r.status_code}")
    if r.status_code == 200:
        print(r.text[:500])
except Exception as e:
    print(f"  오류: {e}")

# 2. 관세청 HS코드 API (품목분류)
print("\n=== 관세청 품목분류 API ===")
try:
    url2 = "https://apis.data.go.kr/1220000/tariffHsService/getHsList"
    params2 = {
        "serviceKey": API_KEY,
        "type": "json",
        "numOfRows": 5,
        "pageNo": 1,
        "query": "화장품",
    }
    r2 = requests.get(url2, params=params2, timeout=8)
    print(f"관세청 품목분류 → {r2.status_code}")
    if r2.status_code == 200:
        print(r2.text[:500])
except Exception as e:
    print(f"  오류: {e}")

# 3. 한국무역통계진흥원 수출통계
print("\n=== 한국무역통계진흥원 API ===")
try:
    url3 = "https://apis.data.go.kr/B552582/ktnet01/getTrdstatsExptList"
    params3 = {
        "serviceKey": API_KEY,
        "numOfRows": 5,
        "pageNo": 1,
        "type": "json",
    }
    r3 = requests.get(url3, params=params3, timeout=8)
    print(f"무역통계진흥원 수출 → {r3.status_code}")
    if r3.status_code == 200:
        print(r3.text[:300])
except Exception as e:
    print(f"  오류: {e}")

# 4. KOTRA 해외시장 뉴스 API
print("\n=== KOTRA 해외시장뉴스 API ===")
try:
    url4 = "https://apis.data.go.kr/B410001/ovseaMarket/getOvseaMarketList"
    params4 = {
        "serviceKey": API_KEY,
        "numOfRows": 3,
        "pageNo": 1,
        "type": "json",
    }
    r4 = requests.get(url4, params=params4, timeout=8)
    print(f"KOTRA 해외시장뉴스 → {r4.status_code}")
    if r4.status_code == 200:
        print(r4.text[:500])
except Exception as e:
    print(f"  오류: {e}")


=== 관세청 수출입 통계 API 테스트 ===


관세청 무역통계 → 500

=== 관세청 품목분류 API ===


관세청 품목분류 → 500

=== 한국무역통계진흥원 API ===


무역통계진흥원 수출 → 500

=== KOTRA 해외시장뉴스 API ===


KOTRA 해외시장뉴스 → 500


In [14]:

import requests

API_KEY = "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23"

# 실제 작동하는 API만 찾기
test_apis = [
    # KOTRA 계열
    ("KOTRA 수출지원기반활용사업 신청기업", "https://apis.data.go.kr/B410001/exportSpprtBizInfo/getExportSpprtBizList"),
    ("KOTRA 해외시장뉴스", "https://apis.data.go.kr/B410001/ovseaMarketInfo/getOvseaMarketInfo"),
    ("KOTRA 국가정보", "https://apis.data.go.kr/B410001/countryInfo/getCountryInfoList"),
    # K-SURE
    ("K-SURE 수출보험 통계", "https://apis.data.go.kr/B552696/exportInsuranceStats/getExportInsuranceStatsList"),
    ("K-SURE 국가위험도", "https://apis.data.go.kr/B552696/countryRisk/getCountryRiskList"),
    # 관세청
    ("관세청 수출입통계", "https://apis.data.go.kr/1220000/salesStatisticsService/getSalesStatisticsList"),
    ("관세청 FTA협정세율", "https://apis.data.go.kr/1220000/ftaTariffService/getFtaTariffList"),
    ("관세청 세율정보", "https://apis.data.go.kr/1220000/tariffService/getTariffList"),
    # 중진공
    ("중진공 수출지원사업", "https://apis.data.go.kr/B552843/exportSupportService/getExportSupportList"),
    # 식약처
    ("식약처 화장품 수출실적", "https://apis.data.go.kr/1471000/CosmeticsExportList/getCosmeticsExportList"),
    ("식약처 화장품 인증", "https://apis.data.go.kr/1471000/CosmeticsPermitListService/getCosmeticsPermitList"),
    # aT
    ("aT 수출통계", "https://apis.data.go.kr/B552745/aT_tradeStat/getAT_tradeStatList"),
    # 무역협회
    ("무역협회 수출통계", "https://apis.data.go.kr/B460016/tradeStatsService/getTradeStatsList"),
]

results = []
for name, url in test_apis:
    try:
        params = {"serviceKey": API_KEY, "numOfRows": "1", "pageNo": "1", "type": "json"}
        r = requests.get(url, params=params, timeout=6)
        status = r.status_code
        snippet = r.text[:200].replace('\n','')
        results.append((status, name, url, snippet))
        print(f"[{status}] {name}")
        if status == 200:
            print(f"  ✅ 응답: {snippet[:150]}")
    except Exception as e:
        print(f"[ERR] {name}: {e}")

print(f"\n성공(200): {sum(1 for s,*_ in results if s==200)}/{len(results)}")
print(f"실패(500): {sum(1 for s,*_ in results if s==500)}/{len(results)}")


[500] KOTRA 수출지원기반활용사업 신청기업


[500] KOTRA 해외시장뉴스


[500] KOTRA 국가정보


[500] K-SURE 수출보험 통계


[500] K-SURE 국가위험도


...

[500] 중진공 수출지원사업


[500] 식약처 화장품 수출실적


[500] 식약처 화장품 인증


[500] aT 수출통계


[500] 무역협회 수출통계

성공(200): 0/13
실패(500): 13/13


In [17]:

import requests

API_KEY = "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23"

# NIPA 계열 더 탐색
nipa_apis = [
    ("NIPA 글로벌ICT포털 해외전시회", "https://apis.data.go.kr/B552551/exhibitionList/getExhibitionList"),
    ("NIPA 글로벌ICT 해외기업", "https://apis.data.go.kr/B552551/overseasCompanyList/getOverseasCompanyList"),
    ("NIPA 글로벌ICT 사업기회", "https://apis.data.go.kr/B552551/bizOpportunityList/getBizOpportunityList"),
    ("NIPA ICT 수출상담회", "https://apis.data.go.kr/B552551/consultationList/getConsultationList"),
]

# K-SURE 계열
ksure_apis = [
    ("K-SURE 수출실적통계", "https://apis.data.go.kr/B552696/statExportInsurance/getStatExportInsuranceList"),
    ("K-SURE 국가정보", "https://apis.data.go.kr/B552696/countryInfo/getCountryInfoList"),
    ("K-SURE 바이어신용조사", "https://apis.data.go.kr/B552696/creditInfo/getCreditInfoList"),
    ("K-SURE 보험가입기업", "https://apis.data.go.kr/B552696/insuredCompany/getInsuredCompanyList"),
]

# 관세청 정식 API
customs_apis = [
    ("관세청 HS품목코드조회", "https://unipass.customs.go.kr/openapi/rest/tariffService/hs/search"),
    ("관세청 FTA세율", "https://unipass.customs.go.kr/openapi/rest/tariffService/ftaTariff/search"),
    ("관세법령정보포털 HS", "https://apis.data.go.kr/1220000/tariffHsCode/getTariffHsCodeList"),
]

all_apis = nipa_apis + ksure_apis + customs_apis

working = []
for name, url in all_apis:
    try:
        params = {"serviceKey": API_KEY, "numOfRows": "3", "pageNo": "1", "type": "json"}
        r = requests.get(url, params=params, timeout=7)
        code = r.status_code
        snippet = r.text[:300]
        if code == 200:
            working.append((name, url, snippet))
            print(f"✅ [{code}] {name}")
            print(f"   {snippet[:200]}\n")
        else:
            print(f"❌ [{code}] {name}")
    except Exception as e:
        print(f"⚠️  [ERR] {name}: {e}")

print(f"\n작동: {len(working)}/{len(all_apis)}")


❌ [500] NIPA 글로벌ICT포털 해외전시회


❌ [500] NIPA 글로벌ICT 해외기업


❌ [500] NIPA 글로벌ICT 사업기회


❌ [500] NIPA ICT 수출상담회


❌ [500] K-SURE 수출실적통계


...

❌ [500] K-SURE 바이어신용조사


❌ [500] K-SURE 보험가입기업


⚠️  [ERR] 관세청 HS품목코드조회: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


✅ [200] 관세청 FTA세율
   



❌ [500] 관세법령정보포털 HS

작동: 1/11


In [20]:

import requests, json

API_KEY = "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23"

# 관세청 FTA세율 API 상세 탐색
url = "https://unipass.customs.go.kr/openapi/rest/tariffService/ftaTariff/search"

# 화장품 HS 330499로 조회
params = {
    "serviceKey": API_KEY,
    "hsCd": "330499",
    "lang": "ko",
    "numOfRows": 10,
    "pageNo": 1,
}
r = requests.get(url, params=params, timeout=10)
print(f"상태: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type','')}")
print(f"응답길이: {len(r.text)}")
print(r.text[:1000])

# XML인 경우 파싱
if 'xml' in r.headers.get('Content-Type','').lower() or r.text.strip().startswith('<'):
    import xml.etree.ElementTree as ET
    try:
        root = ET.fromstring(r.text)
        print("\n=== XML 파싱 ===")
        for child in root.iter():
            if child.text and child.text.strip():
                print(f"  <{child.tag}>: {child.text.strip()[:80]}")
    except:
        pass


ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

In [23]:

import requests

API_KEY = "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23"

# 관세청 국가별 수출입실적 GW API (실제 운영 중인 것)
print("=== 관세청 품목별 국가별 수출입실적 ===")
try:
    url = "https://apis.data.go.kr/1220000/itemCountryService1/getitemCountryList1"
    params = {
        "serviceKey": API_KEY,
        "type": "json",
        "numOfRows": "5",
        "pageNo": "1",
        "year": "2025",
        "catNo": "3304",   # 화장품 HS
        "term": "MT",      # Monthly
    }
    r = requests.get(url, params=params, timeout=10)
    print(f"상태: {r.status_code}")
    print(r.text[:500])
except Exception as e:
    print(f"오류: {e}")

# 관세청 국가별 수출입실적
print("\n=== 관세청 국가별 수출입실적 ===")
try:
    url2 = "https://apis.data.go.kr/1220000/userCountryService1/getUserCountryList1"
    params2 = {
        "serviceKey": API_KEY,
        "type": "json",
        "numOfRows": "5",
        "pageNo": "1",
        "year": "2025",
    }
    r2 = requests.get(url2, params=params2, timeout=10)
    print(f"상태: {r2.status_code}")
    print(r2.text[:500])
except Exception as e:
    print(f"오류: {e}")

# bizinfo 중소벤처기업부 지원사업 API
print("\n=== 중소벤처기업부 bizinfo 지원사업 API ===")
try:
    url3 = "https://www.bizinfo.go.kr/uss/rss/bizinfoApi.do"
    params3 = {
        "crtfcKey": API_KEY,
        "dataType": "json",
        "searchCnt": "5",
        "searchLclasId": "CCRS000017",  # 수출분야
    }
    r3 = requests.get(url3, params=params3, timeout=10)
    print(f"상태: {r3.status_code}")
    print(r3.text[:500])
except Exception as e:
    print(f"오류: {e}")

# 식약처 화장품 원료성분
print("\n=== 식약처 화장품 원료성분 API ===")
try:
    url4 = "https://apis.data.go.kr/1471000/CosmeticService/getCosmeticIngrdntList"
    params4 = {
        "serviceKey": API_KEY,
        "type": "json",
        "numOfRows": "3",
        "pageNo": "1",
        "ingrdntKorNm": "에탄올",
    }
    r4 = requests.get(url4, params=params4, timeout=10)
    print(f"상태: {r4.status_code}")
    print(r4.text[:400])
except Exception as e:
    print(f"오류: {e}")


=== 관세청 품목별 국가별 수출입실적 ===


상태: 500
Unexpected errors


=== 관세청 국가별 수출입실적 ===


상태: 500
Unexpected errors


=== 중소벤처기업부 bizinfo 지원사업 API ===


오류: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

=== 식약처 화장품 원료성분 API ===


상태: 500
Unexpected errors



In [26]:

import requests
import socket

# 현재 서버 IP 확인
print("=== 서버 환경 확인 ===")
try:
    r = requests.get("https://api.ipify.org?format=json", timeout=5)
    print(f"외부 IP: {r.json().get('ip','?')}")
except:
    print("IP 확인 실패")

# 해외 API 정상 작동 확인
print("\n=== 해외 API 테스트 (K-SURE - 기존 작동 확인) ===")
try:
    url = "https://apis.data.go.kr/B552696/getBuyerList/getBuyerList"
    params = {
        "serviceKey": "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23",
        "ctryCd": "450",
        "prodNm": "cosmetic",
        "pageNo": "1",
        "numOfRows": "3",
        "type": "json",
    }
    r = requests.get(url, params=params, timeout=10)
    print(f"K-SURE 상태: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"✅ K-SURE 정상 - 총: {data.get('totalCount', '?')}건")
    else:
        print(r.text[:200])
except Exception as e:
    print(f"K-SURE 오류: {e}")

# NIPA API 테스트
print("\n=== NIPA ICT 바이어 API (기존 작동 확인) ===")
try:
    url2 = "https://apis.data.go.kr/B552551/overseasBuyerList/getOverseasBuyerList"
    params2 = {
        "serviceKey": "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23",
        "numOfRows": "3",
        "pageNo": "1",
        "type": "json",
    }
    r2 = requests.get(url2, params=params2, timeout=10)
    print(f"NIPA 상태: {r2.status_code}")
    if r2.status_code == 200:
        print(f"✅ NIPA 정상 - {r2.text[:200]}")
    else:
        print(r2.text[:200])
except Exception as e:
    print(f"NIPA 오류: {e}")

print("\n=== 결론: 서버에서 공공데이터포털 API 접근 제한 분석 ===")
print("K-SURE(B552696) → 정상")
print("NIPA(B552551) → 정상")
print("관세청(1220000) → 500 오류")
print("식약처(1471000) → 500 오류")
print("KOTRA(B410001) → 500 오류")
print("→ IP 화이트리스트 또는 기관별 별도 API키 필요 가능성")


=== 서버 환경 확인 ===
외부 IP: 47.252.83.71

=== 해외 API 테스트 (K-SURE - 기존 작동 확인) ===


K-SURE 상태: 500
Unexpected errors


=== NIPA ICT 바이어 API (기존 작동 확인) ===


NIPA 상태: 200
✅ NIPA 정상 - {"response":{"header":{"resultCode":200,"resultMsg":"Success"},"body":{"items":[{"buyNo":4101,"buyName":"PS Cloud Services ","nationName":"카자흐스탄","phone":"(+7) (727) 388 80 00","regDateStr":"2025.12.0

=== 결론: 서버에서 공공데이터포털 API 접근 제한 분석 ===
K-SURE(B552696) → 정상
NIPA(B552551) → 정상
관세청(1220000) → 500 오류
식약처(1471000) → 500 오류
KOTRA(B410001) → 500 오류
→ IP 화이트리스트 또는 기관별 별도 API키 필요 가능성


In [29]:

import requests, time

API_KEY = "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23"

# K-SURE 재시도 (잠시 대기 후)
time.sleep(2)
print("=== K-SURE API 재시도 ===")
try:
    url = "https://apis.data.go.kr/B552696/getBuyerList/getBuyerList"
    params = {
        "serviceKey": API_KEY,
        "ctryCd": "450",
        "prodNm": "cosmetic",
        "pageNo": "1",
        "numOfRows": "3",
        "type": "json",
    }
    r = requests.get(url, params=params, timeout=15)
    print(f"상태: {r.status_code}")
    print(r.text[:500])
except Exception as e:
    print(f"오류: {e}")

# 관세청 실시간 국가별 수출입 다른 endpoint
print("\n=== 관세청 수출실적 다른 endpoint ===")
try:
    # 관세청 화물통관진행정보 (B/L 기반) - 실제 파라미터 필요없는 것
    url2 = "https://apis.data.go.kr/1220000/custborderpassService/getCustborderpassList"
    params2 = {
        "serviceKey": API_KEY,
        "numOfRows": "3",
        "pageNo": "1",
        "blNo": "AMFU2023052900001",  # 예시 B/L 번호
    }
    r2 = requests.get(url2, params=params2, timeout=10)
    print(f"상태: {r2.status_code}")
    print(r2.text[:300])
except Exception as e:
    print(f"오류: {e}")

# 관세청 FTA 다시 (HTTP)
print("\n=== 관세청 FTA API HTTP로 재시도 ===")
try:
    url3 = "http://unipass.customs.go.kr/openapi/rest/tariffService/ftaTariff/search"
    params3 = {
        "serviceKey": API_KEY,
        "hsCd": "330499",
        "lang": "ko",
    }
    r3 = requests.get(url3, params=params3, timeout=10)
    print(f"상태: {r3.status_code}")
    print(f"응답: {r3.text[:500]}")
except Exception as e:
    print(f"오류: {e}")


=== K-SURE API 재시도 ===


상태: 500
Unexpected errors


=== 관세청 수출실적 다른 endpoint ===


상태: 500
Unexpected errors


=== 관세청 FTA API HTTP로 재시도 ===


오류: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


In [32]:

import requests

API_KEY = "83b96790de580e57527e049d59bfcb18ae34d2bfe646c11a5d2ee6b3d95e9b23"

# 공공데이터포털 HTTP 연결 자체는 가능한지 확인
print("=== 공공데이터포털 HTTP 연결 테스트 ===")
try:
    r = requests.get("https://www.data.go.kr", timeout=5)
    print(f"공공데이터포털 메인 접속: {r.status_code}")
except Exception as e:
    print(f"메인 접속 오류: {e}")

# 일반 외부 API 접근 테스트
try:
    r2 = requests.get("https://restcountries.com/v3.1/alpha/KR", timeout=5)
    print(f"RestCountries API: {r2.status_code}")
except Exception as e:
    print(f"오류: {e}")

# Exchange Rate API
try:
    r3 = requests.get("https://open.er-api.com/v6/latest/USD", timeout=5)
    data3 = r3.json()
    print(f"환율 API: {r3.status_code} - KRW: {data3.get('rates',{}).get('KRW','?')}")
except Exception as e:
    print(f"오류: {e}")

print("""
=== 분석 결과 및 전략 정리 ===

현재 상황:
- NIPA API (B552551): ✅ 정상 작동
- K-SURE API (B552696): ⚠️  현재 일시적 500 오류 (이전 세션에서는 정상)
- 관세청 (1220000): ❌ 500 오류 지속
- 식약처 (1471000): ❌ 500 오류 지속
- KOTRA (B410001): ❌ 500 오류 지속
- unipass.customs.go.kr: ❌ 연결 거부

원인 분석:
1. 서버 IP(47.252.83.71)가 공공데이터포털 API 서버 IP 허용 목록에 없을 가능성
2. 기관별로 별도 신청/승인이 필요한 API일 가능성
3. API 키가 특정 기관 API에만 등록되어 있을 가능성
4. 현재 서버(클라우드 샌드박스)에서 정부기관 API 접근 차단

보완 방향:
1. 현재 작동하는 API: NIPA, K-SURE (간헐적) → 계속 활용
2. 관세청 수출통계: CSV 파일로 직접 다운로드하여 활용
3. KOTRA 해외시장뉴스: data.go.kr에서 파일 데이터셋 다운로드 활용
4. 식약처 화장품 규제정보: 정적 CSV로 구성 가능

현재 시스템에 통합된 실제 데이터 요약:
- buyer_db.csv: 46,089행 (KOTRA SNS 실데이터)
- kotra_sns_buyers.csv: 46,034행
- kotra_inquiry.csv: 40,305행 (HS코드 추정 추가)
- kotra_buyer_stats.csv: 3,065행
- smba_inquiry.csv: 21,302행 (HS코드 추정 추가)
- smba_purchase_offer.csv: 326행
- nipa_ict_buyers.csv: 1,853행 (ISO코드 추가)
- ksure_cosmetic_email_verified.csv: 214행 (국가 ISO 추가)
- ksure_cosmetic_buyers_full.csv: 386행
- aT_bms_buyers.csv: 5,435행
- trade_regulation_db.csv: 27,959행
- country_credit_db.csv: 195행
- email_pattern_db.csv: ~50행
- kotra_hs_country_recommend.csv: 2,100행 (ISO코드 보완 76.3%)

총 데이터: ~195,000+ 행 실제 공공데이터
""")


=== 공공데이터포털 HTTP 연결 테스트 ===


메인 접속 오류: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
RestCountries API: 200
환율 API: 200 - KRW: 1501.222192

=== 분석 결과 및 전략 정리 ===

현재 상황:
- NIPA API (B552551): ✅ 정상 작동
- K-SURE API (B552696): ⚠️  현재 일시적 500 오류 (이전 세션에서는 정상)
- 관세청 (1220000): ❌ 500 오류 지속
- 식약처 (1471000): ❌ 500 오류 지속
- KOTRA (B410001): ❌ 500 오류 지속
- unipass.customs.go.kr: ❌ 연결 거부

원인 분석:
1. 서버 IP(47.252.83.71)가 공공데이터포털 API 서버 IP 허용 목록에 없을 가능성
2. 기관별로 별도 신청/승인이 필요한 API일 가능성
3. API 키가 특정 기관 API에만 등록되어 있을 가능성
4. 현재 서버(클라우드 샌드박스)에서 정부기관 API 접근 차단

보완 방향:
1. 현재 작동하는 API: NIPA, K-SURE (간헐적) → 계속 활용
2. 관세청 수출통계: CSV 파일로 직접 다운로드하여 활용
3. KOTRA 해외시장뉴스: data.go.kr에서 파일 데이터셋 다운로드 활용
4. 식약처 화장품 규제정보: 정적 CSV로 구성 가능

현재 시스템에 통합된 실제 데이터 요약:
- buyer_db.csv: 46,089행 (KOTRA SNS 실데이터)
- kotra_sns_buyers.csv: 46,034행
- kotra_inquiry.csv: 40,305행 (HS코드 추정 추가)
- kotra_buyer_stats.csv: 3,065행
- smba_inquiry.csv: 21,302행 (HS코드 추정 추가)
- smba_purchase_offer.csv: 326행
- nipa_ict_buyers.csv: 1,853행 (ISO코드 추가)
- ksu